In [ ]:
using MPSDynamics, ChaosTools, ProgressBars
include("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/config/requirements.jl");

In [ ]:
const ∞ = Inf
ħ=1
e=1
β = ∞ # Case zero temperature T=0, β → ∞
ϕ_quantum=1/2

#### Caracterization of the circuit #####

ω_a = 25.338776456203686
ω_b_0 = 2*ω_a

Lamb_shift = 0.05 #Lamb shift in GHz


φ_a = 0.11#*(rand(1)[1]/2+0.75)
Z_a = 2*ϕ_quantum^2*φ_a^2
C_a_inv = Z_a*ω_a
E_Ca = C_a_inv/2

φ_b = 0.204#*(rand(1)[1]/2+0.75)
Z_b = 2*ϕ_quantum^2*φ_b^2
C_b_inv = Z_b*ω_b_0
E_Cb = C_b_inv/2

E_J = 37.12*2*pi*1.5#*(rand(1)[1]/2+0.75)


##Drive


ω_d=ω_b_0#+Lamb_shift*2*pi # Drive frequency
ω_p=2ω_a-ω_d

ϵ_0 = 0.4581818181818182
g2 = besselj(1,ϵ_0)*E_J*φ_a^2*φ_b # (Leon's paper)
ϵ_d=10.124276*g2

buffer_phase = 0.0
pump_phase = 0.0;

##Env

κ_b=1/10.4#*(rand(1)[1]/2+0.75); # 1/ns from Leon's paper
κ_2 = 4*g2^2/κ_b
κ_a = 0#3e-3*κ_2 #from Leon's paper

In [ ]:
t_max = 10/κ_b
re_a_0, im_a_0 = 0, 0
ϕ_a_0, q_a_0=sqrt(2*Z_a)*re_a_0, sqrt(2/Z_a)*im_a_0
re_b_0, im_b_0 = 0, 0
ϕ_b_0, q_b_0=sqrt(2*Z_b)*re_b_0, sqrt(2/Z_b)*im_b_0

system = SystemATS(ω_a, ω_b_0, E_J, E_Ca, E_Cb, ϕ_a_0, q_a_0, ϕ_b_0, q_b_0)
environment = MeanField(κ_a, κ_b) #OneChain(n, chain_params...)

system_prepared = prepare(system)
environment_prepared = prepare(system,environment)

dt = get_dt(system, environment)
u_dot_0, u_0 = get_init(system_prepared, environment_prepared)
u_0_new = vcat(u_0,u_dot_0);

In [ ]:
ϵ_p_list = vcat(0.2:.025:1.1)
ϵ_d_g2_ratio_list = 4:.2:13.5

In [ ]:
Lyap_coef_chaos = Array{Float64}(undef, length(ϵ_d_g2_ratio_list), length(ϵ_p_list))
drive = DriveTwoModesJA(0.0, ω_d, buffer_phase, 0.0, ω_p, pump_phase)
drive_prepared = prepare(drive)
t_w = 30/κ_b
for i in ProgressBar(1:length(ϵ_p_list))
    ϵ_0_i = ϵ_p_list[i]
    drive.ϵ_0 = ϵ_0_i
    g2_max_i = besselj(1,ϵ_0_i)*E_J*φ_a^2*φ_b # (Leon's paper)
    for j in 1:length(ϵ_d_g2_ratio_list)
        ϵ_d_j = ϵ_d_g2_ratio_list[j]*g2_max_i
        drive.ϵ_d = ϵ_d_j
        drive_prepared.drive = drive
        p = SimulationParams(system_prepared, drive_prepared, environment_prepared)

        problem = CoupledODEs(equation_1ODE!, u_0_new, p)

        Lyap_coef_chaos[j,i] = lyapunov(problem, t_max, Δt=dt; d0 = 1e-7, d0_upper = 1e-4, Ttr = t_w)
    end
end

In [ ]:
Lyap_coef_chaos_sign = map(x -> x>0 ? 1 : -1, Lyap_coef_chaos);

In [ ]:
using JSON
ϵ_p_zero_list = JSON.parsefile("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/analysis/output/epsilon_zero_list_05_02_2026.json")
ϵ_p_zero_list_approx = JSON.parsefile("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/analysis/output/epsilon_zero_list_05_02_2026_approx.json");
ϵ_p_zero_list_Hill = JSON.parsefile("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/analysis/output/epsilon_zero_list_12_02_2026_Hill.json");
ϵ_p_zero_list_Hill_unstable = JSON.parsefile("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/analysis/output/epsilon_zero_list_12_02_2026_Hill_unstable.json");
α_sq_Hill = range(4,13.5,20)./2;

In [ ]:
α_sq= (ϵ_d_g2_ratio_list./2)
g2_list = E_J.*besselj.(1,ϵ_p_list).*φ_a^2 .*φ_b
g2_zero_list = E_J.*besselj.(1,ϵ_p_zero_list).*φ_a^2 .*φ_b
g2_zero_list_approx = E_J.*besselj.(1,ϵ_p_zero_list_approx).*φ_a^2 .*φ_b
g2_zero_list_Hill = E_J.*besselj.(1,ϵ_p_zero_list_Hill).*φ_a^2 .*φ_b
g2_zero_list_Hill_unstable = E_J.*besselj.(1,ϵ_p_zero_list_Hill_unstable).*φ_a^2 .*φ_b
heatmap(g2_list/2π*1e3,α_sq, Lyap_coef_chaos_sign, cmap=:bluesreds)
plot!(g2_zero_list_approx/2π*1e3, α_sq, label = "Approx threshold period doubling")
plot!(g2_zero_list/2π*1e3, α_sq, label = "Refined threshold period doubling")
plot!(g2_zero_list_Hill/2π*1e3, α_sq_Hill, label = "Hill's period doubling")
#plot!(g2_zero_list_Hill_unstable/2π*1e3, α_sq_Hill, label = "Hill's unstability")
plot!(xlabel=L"g_2/2\pi"*" (MHz)", ylabel=L"|\alpha_{th}|^2",
    size=(size_px_car[1],size_px_car[2]),
    xlims=(minimum(g2_list/2π*1e3),maximum(g2_list/2π*1e3)),ylims=(minimum(α_sq),maximum(α_sq)+2),
    colorbar_title="\nSign of Lyapunov exp.", title = "Mean field "*L"κ_b t= "*"$(κ_b*t_max)",
    clims=(minimum(Lyap_coef_chaos_sign),maximum(Lyap_coef_chaos_sign)),
    right_margin=10mm)
#savefig("../../figures/chaos/heatmap_lyapunov_coef_sign_omega_p_0.pdf")

In [ ]:
ϵ_p_list = vcat(0.55:.01:0.8)
ϵ_d_g2_ratio_list = 4:.2:13.5
initial_values = JSON.parsefile("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/analysis/output/initial_values_12_02_2026.json");

In [ ]:
Lyap_coef_chaos = Array{Float64}(undef, length(ϵ_d_g2_ratio_list), length(ϵ_p_list))
drive = DriveTwoModesJA(0.0, ω_d, buffer_phase, 0.0, ω_p, pump_phase)
drive_prepared = prepare(drive)
t_w = 30/κ_b
for i in ProgressBar(1:length(ϵ_p_list))
    ϵ_0_i = ϵ_p_list[i]
    drive.ϵ_0 = ϵ_0_i
    g2_max_i = besselj(1,ϵ_0_i)*E_J*φ_a^2*φ_b # (Leon's paper)
    for j in 1:length(ϵ_d_g2_ratio_list)
        re_a_0, im_a_0 = initial_values[j]
        ϕ_a_0, q_a_0=sqrt(2*Z_a)*re_a_0, sqrt(2/Z_a)*im_a_0
        re_b_0, im_b_0 = 0, 0
        ϕ_b_0, q_b_0=sqrt(2*Z_b)*re_b_0, sqrt(2/Z_b)*im_b_0

        system = SystemATS(ω_a, ω_b_0, E_J, E_Ca, E_Cb, ϕ_a_0, q_a_0, ϕ_b_0, q_b_0)
        environment = MeanField(κ_a, κ_b) #OneChain(n, chain_params...)

        system_prepared = prepare(system)
        environment_prepared = prepare(system,environment)

        dt = get_dt(system, environment)
        u_dot_0, u_0 = get_init(system_prepared, environment_prepared)
        u_0_new = vcat(u_0,u_dot_0);
        ϵ_d_j = ϵ_d_g2_ratio_list[j]*g2_max_i
        drive.ϵ_d = ϵ_d_j
        drive_prepared.drive = drive
        p = SimulationParams(system_prepared, drive_prepared, environment_prepared)

        problem = CoupledODEs(equation_1ODE!, u_0_new, p)

        Lyap_coef_chaos[j,i] = lyapunov(problem, t_max, Δt=dt; d0 = 1e-7, d0_upper = 1e-4, Ttr = t_w)
    end
end
Lyap_coef_chaos_sign = map(x -> x>0 ? 1 : -1, Lyap_coef_chaos);

In [ ]:
α_sq= (ϵ_d_g2_ratio_list./2)
g2_list = E_J.*besselj.(1,ϵ_p_list).*φ_a^2 .*φ_b
g2_zero_list = E_J.*besselj.(1,ϵ_p_zero_list).*φ_a^2 .*φ_b
g2_zero_list_approx = E_J.*besselj.(1,ϵ_p_zero_list_approx).*φ_a^2 .*φ_b
g2_zero_list_Hill = E_J.*besselj.(1,ϵ_p_zero_list_Hill).*φ_a^2 .*φ_b
g2_zero_list_Hill_unstable = E_J.*besselj.(1,ϵ_p_zero_list_Hill_unstable).*φ_a^2 .*φ_b
heatmap(g2_list/2π*1e3,α_sq, Lyap_coef_chaos_sign, cmap=:bluesreds)
#plot!(g2_zero_list_approx/2π*1e3, α_sq, label = "Approx threshold period doubling")
#plot!(g2_zero_list/2π*1e3, α_sq, label = "Refined threshold period doubling")
plot!(g2_zero_list_Hill/2π*1e3, α_sq_Hill, label = "Hill's Period doubling")
plot!(g2_zero_list_Hill_unstable/2π*1e3, α_sq_Hill, label = "Hill's unstability")
plot!(xlabel=L"g_2/2\pi"*" (MHz)", ylabel=L"|\alpha_{th}|^2",
    size=(size_px_car[1],size_px_car[2]),
    xlims=(minimum(g2_list/2π*1e3),maximum(g2_list/2π*1e3)),ylims=(minimum(α_sq_Hill),maximum(α_sq)+2),
    colorbar_title="\nSign of Lyapunov exp.", title = "Mean field "*L"κ_b t= "*"$(κ_b*t_max)",
    clims=(minimum(Lyap_coef_chaos_sign),maximum(Lyap_coef_chaos_sign)),
    right_margin=10mm)
#savefig("../../figures/chaos/heatmap_lyapunov_coef_sign_omega_p_0.pdf")